<a href="https://colab.research.google.com/github/lorek/RPAI_lab/blob/main/Lista_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Paweł Lorek  
University of Wrocław

# Lista 1. Od losowania do prawdopodobieństwa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)

## 1. Dwie uczciwe kostki
**Przewidywanie:** które sumy od 2 do 12 pojawią się najczęściej? Czy suma 12 jest równie prawdopodobna jak 7?
Zakładamy niezależne rzuty dwiema uczciwymi kostkami. Jeden wiersz to jedno doświadczenie.

In [ ]:
n = 10_000
dice = rng.integers(1, 7, size=(n, 2))
totals = dice.sum(axis=1)
print(dice[:5])
print("Sumy:", totals[:5])
print("Częstość sumy 7:", (totals == 7).mean())

In [ ]:
sums = np.arange(2, 13)
frequencies = np.array([(totals == s).mean() for s in sums])
plt.figure(figsize=(8, 3))
plt.bar(sums, frequencies)
plt.xticks(sums)
plt.xlabel("Suma oczek")
plt.ylabel("Częstość")
plt.show()

**Zadanie 1.1.** Zapisz maski i oszacuj szanse:
1. sumy podzielnej przez 5;
2. co najmniej jednej szóstki;
3. dokładnie jednej szóstki.

Najpierw dla par (6,6), (6,2), (2,6), (2,3) ustal ręcznie, które zdarzenia zachodzą. „Co najmniej jedna” obejmuje dwie szóstki, „dokładnie jedna” ich nie obejmuje.

In [ ]:
# TODO: Zapisz trzy zdarzenia jako maski i oblicz ich częstości.

## 2. Mały model, który możemy policzyć dokładnie
Istnieje 36 uporządkowanych par, wszystkie jednakowo prawdopodobne. (1,6) i (6,1) to różne wyniki.
Poniższa tablica zawiera wszystkie pary, a nie losową próbę.

In [ ]:
pairs = np.array([(a, b) for a in range(1, 7) for b in range(1, 7)])
exact_totals = pairs.sum(axis=1)
exact_probabilities = np.array([(exact_totals == s).mean() for s in sums])
plt.figure(figsize=(8, 3))
plt.bar(sums, frequencies, alpha=0.6, label="Symulacja")
plt.plot(sums, exact_probabilities, "o", color="black", label="Dokładny wynik")
plt.xticks(sums)
plt.xlabel("Suma oczek")
plt.ylabel("Prawdopodobieństwo / częstość")
plt.legend()
plt.show()

**Zadanie 1.2.** Oblicz dokładne prawdopodobieństwa trzech zdarzeń z zadania 1.1, używając `pairs`. Porównaj z symulacją.
Czy różnica oznacza błąd kodu? Powtórz dla 100 i 100 000 doświadczeń.

**Wniosek do zapisania:** dlaczego jednakowo prawdopodobne rzuty nie dają jednakowo prawdopodobnych sum?

In [ ]:
# TODO: Oblicz prawdopodobieństwa na podstawie wszystkich 36 par i porównaj z symulacją.

## 3. Paradoks urodzin
**Przewidywanie:** ile osób wystarczy, aby szansa wspólnych urodzin przekroczyła 1/2?
Chodzi o **dowolną parę osób**, nie o osobę mającą urodziny tego samego dnia co Ty.

Model: 365 dni, jednakowo prawdopodobnych; urodziny różnych osób niezależne; pomijamy 29 lutego. To uproszczenie rzeczywistości.
Najpierw jedna grupa: dzień kodujemy liczbą 0,…,364.

In [ ]:
group = rng.integers(0, 365, size=23)
print("Dni urodzin:", group)
print("Różne dni:", np.unique(group).size)
print("Czy jest wspólna data?", np.unique(group).size < group.size)

Teraz powtarzamy doświadczenie. Każdy wiersz jest nową grupą. Po uporządkowaniu dni wspólna data oznacza równe sąsiednie elementy. `any(axis=1)` sprawdza, czy w danym wierszu jest choć jedna taka para.

In [ ]:
n_groups = 10_000
group_size = 23
birthdays = rng.integers(0, 365, size=(n_groups, group_size))
ordered = np.sort(birthdays, axis=1)
shared = (ordered[:, 1:] == ordered[:, :-1]).any(axis=1)
print("Oszacowanie:", shared.mean())

**Zadanie 1.3.** Powtórz dla grup 10, 20, 23, 30 i 50 osób. Zapisz wielkości grup i oszacowania; narysuj wykres punktowy.
Nie losuj bez zwracania — wykluczyłoby to wspólne urodziny z definicji!

In [ ]:
# TODO: Wykonaj symulacje dla pięciu wielkości grup i narysuj wyniki.

## 4. Dlaczego wystarczą tak małe grupy?
Łatwiej policzyć zdarzenie przeciwne: wszyscy mają różne daty.
Pierwsza osoba może mieć dowolną datę. Druga musi uniknąć jednej daty, trzecia dwóch itd.
Dla grupy wielkości k ≤ 365:

$$P(\text{wspólna data})=1-\prod_{j=0}^{k-1}\frac{365-j}{365}.$$

Dla k > 365 wspólna data jest pewna. Czy widzisz dlaczego?

In [ ]:
sizes = np.arange(1, 61)
exact = np.array([1 - np.prod((365 - np.arange(k)) / 365) for k in sizes])
plt.figure(figsize=(7, 3))
plt.plot(sizes, exact, label="Dokładne prawdopodobieństwo")
plt.axhline(0.5, color="black", linestyle="--", label="1/2")
plt.xlabel("Liczba osób")
plt.ylabel("Szansa wspólnych urodzin")
plt.legend()
plt.show()
print("Pierwszy rozmiar z szansą > 1/2:", sizes[exact > 0.5][0])

**Zadanie 1.4.** Dodaj swoje wyniki symulacji do wykresu. Ile par osób można utworzyć w grupie 23 osób? Oblicz `23 * 22 // 2`.
Duża liczba par pomaga zrozumieć zaskoczenie, ale nie oznacza, że zdarzenia dla tych par są niezależne.

**Bonus:** w grupie 23 osób oszacuj szansę, że ktoś inny ma tę samą datę co pierwsza osoba. Porównaj z szansą dowolnej wspólnej daty.

## Na koniec — trzy zdania własnymi słowami
1. Co było jednym doświadczeniem w każdym z dwóch przykładów?
2. Dlaczego częstość może różnić się od prawdopodobieństwa?
3. Które wyniki symulacji różniły się od Twoich przewidywań?